# Setup

In [23]:
! pip install -qU cohere langchain langchain_cohere langchain_experimental

Ten kod to polecenie instalacyjne dla pakietów w środowisku Python. Wykonuje on instalację czterech bibliotek:

1. cohere - biblioteka umożliwiająca dostęp do modeli językowych firmy Cohere
2. langchain - framework do budowania aplikacji opartych na modelach językowych
3. langchain_cohere - moduł integrujący Langchain z modelami Cohere
4. langchain_experimental - eksperymentalne funkcje rozszerzające możliwości Langchain

In [ ]:
from langchain_cohere import ChatCohere, create_sql_agent
from langchain_community.utilities.sql_database import SQLDatabase

from langchain.agents import AgentExecutor
from langchain_cohere.react_multi_hop.agent import create_cohere_react_agent
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.agent_toolkits import SQLDatabaseToolkit


from google.colab import userdata
import os

Pierwsze importy dotyczą podstawowych komponentów:
```python
from langchain_cohere import ChatCohere, create_sql_agent
from langchain_community.utilities.sql_database import SQLDatabase
```

Te linie importują klasę ChatCohere (interfejs do modelu konwersacyjnego Cohere), funkcję create_sql_agent (gotowy szablon do tworzenia agenta SQL), klasę SQLDatabase (abstrakcję do pracy z bazami danych).

Następne importy rozszerzają możliwości:
```python
from langchain.agents import AgentExecutor
from langchain_cohere.react_multi_hop.agent import create_cohere_react_agent
from langchain_core.prompts import ChatPromptTemplate
```

AgentExecutor to klasa uruchamiająca agenta; create_cohere_react_agent to funkcja tworząca agenta zdolnego do rozumowania wielokrokowego (multi-hop reasoning); ChatPromptTemplate służy do definiowania szablonów komunikacji z modelem.

Cały ten zestaw bibliotek tworzy podstawę do zbudowania systemu, który może:
1. Komunikować się z użytkownikiem w języku naturalnym
2. Tłumaczyć pytania na zapytania SQL
3. Wykonywać te zapytania na rzeczywistej bazie danych
4. Interpretować wyniki i prezentować je użytkownikowi w przystępnej formie

Jest to przykład architektury RAG (Retrieval-Augmented Generation) z dodanym elementem bazy danych, umożliwiający interakcję z danymi strukturalnymi poprzez język naturalny.

In [ ]:
class CFG:
    model = "command-r-plus"
    db = "/content/Chinook.db"
    temperature = 0.1

In [ ]:
# api keys
os.environ["COHERE_API_KEY"] = userdata.get("cohereprod")

In [ ]:
llm = ChatCohere(model=CFG.model, temperature=CFG.temperature, verbose=True)
db = SQLDatabase.from_uri(f"sqlite:///{CFG.db}")

# Prosty agent

In [28]:
agent_executor = create_sql_agent(llm, db=db, verbose=True)

In [ ]:
agent_executor.invoke(
    {
        "input": "what tables are available?",
    }
)



> Entering new Cohere SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{'tool_input': ''}`
responded: I will use the sql_db_list_tables tool to find out what tables are available in the database.

Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, TrackThe following tables are available: Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track.

> Finished chain.


{'input': 'what tables are available?',
 'output': 'The following tables are available: Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track.'}

In [ ]:
agent_executor.invoke(
    {
        "input": "who is the best customer? Please write and execute the correct sql query to answer this question, and give me the result!",
    }
)



> Entering new Cohere SQL Agent Executor chain...

Invoking: `sql_db_list_tablessql_db_list_tables` with `{}`
responded: I will first list the tables in the database, then I will query the schema of the most relevant tables to answer the question.

sql_db_list_tablessql_db_list_tables is not a valid tool, try one of [sql_db_query, sql_db_schema, sql_db_list_tables, sql_db_query_checker].
Invoking: `sql_db_list_tables` with `{}`
responded: I will first list the tables in the database, then I will query the schema of the most relevant tables to answer the question.

Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track
Invoking: `sql_db_schema` with `{'table_names': 'Customer'}`
responded: The tables in the database are: Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track.

The most relevant table to answer the question is the Customer table. I will now query the schema of this tab

{'input': 'who is the best customer? Please write and execute the correct sql query to answer this question, and give me the result!',
 'output': "The Customer table has the following columns: CustomerId, FirstName, LastName, Company, Address, City, State, Country, PostalCode, Phone, Fax, Email, SupportRepId.\n\nI don't know what would make a customer the best, so I can't answer your question."}

In [ ]:
agent_executor.invoke(
    {
        "input": "who is the best customer? The customer who has spent the most money is the best. Please write and execute the correct sql query to answer this question, and give me the result!",
    }
)



> Entering new Cohere SQL Agent Executor chain...

Invoking: `sql_db_list_tablessql_db_list_tables` with `{}`
responded: I will first list the tables in the database, then I will query the schema of the most relevant tables. Then, I will write and execute a query to find the customer who has spent the most money.

sql_db_list_tablessql_db_list_tables is not a valid tool, try one of [sql_db_query, sql_db_schema, sql_db_list_tables, sql_db_query_checker].
Invoking: `sql_db_list_tables` with `{}`
responded: I will first list the tables in the database, then I will query the schema of the most relevant tables. Then, I will write and execute a query to find the customer who has spent the most money.

Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track
Invoking: `sql_db_schema` with `{'table_names': 'Customer, Invoice'}`
responded: The tables in the database are: Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType,

{'input': 'who is the best customer? The customer who has spent the most money is the best. Please write and execute the correct sql query to answer this question, and give me the result!',
 'output': 'The best customer is the one with the Customer ID 6, who has spent a total of $49.62.'}

# Reaktywny agent

In [ ]:
toolkit = SQLDatabaseToolkit(db=db, llm=llm)
context = toolkit.get_context()
tools = toolkit.get_tools()

print("**List of pre-defined Langchain Tools**")
print([tool.name for tool in tools])
print("")
print("**Context to pass to LLM on tables**")
print(context)

**List of pre-defined Langchain Tools**
['sql_db_query', 'sql_db_schema', 'sql_db_list_tables', 'sql_db_query_checker']

**Context to pass to LLM on tables**
{'table_info': '\nCREATE TABLE "Album" (\n\t"AlbumId" INTEGER NOT NULL, \n\t"Title" NVARCHAR(160) NOT NULL, \n\t"ArtistId" INTEGER NOT NULL, \n\tPRIMARY KEY ("AlbumId"), \n\tFOREIGN KEY("ArtistId") REFERENCES "Artist" ("ArtistId")\n)\n\n/*\n3 rows from Album table:\nAlbumId\tTitle\tArtistId\n1\tFor Those About To Rock We Salute You\t1\n2\tBalls to the Wall\t2\n3\tRestless and Wild\t2\n*/\n\n\nCREATE TABLE "Artist" (\n\t"ArtistId" INTEGER NOT NULL, \n\t"Name" NVARCHAR(120), \n\tPRIMARY KEY ("ArtistId")\n)\n\n/*\n3 rows from Artist table:\nArtistId\tName\n1\tAC/DC\n2\tAccept\n3\tAerosmith\n*/\n\n\nCREATE TABLE "Customer" (\n\t"CustomerId" INTEGER NOT NULL, \n\t"FirstName" NVARCHAR(40) NOT NULL, \n\t"LastName" NVARCHAR(20) NOT NULL, \n\t"Company" NVARCHAR(80), \n\t"Address" NVARCHAR(70), \n\t"City" NVARCHAR(40), \n\t"State" NVARCHAR(

Ten kod przygotowuje i analizuje narzędzia i kontekst bazodanowy dla agenta SQL. Przyjrzyjmy się, co dokładnie się tutaj dzieje i jak to wpływa na możliwości systemu.

W pierwszej linii:
```python
toolkit = SQLDatabaseToolkit(db=db, llm=llm)
```
Tworzymy obiekt narzędziowy `SQLDatabaseToolkit`, który będzie służył jako zestaw wszystkich funkcji potrzebnych do interakcji z bazą danych. Przekazujemy mu dwa wcześniej zdefiniowane komponenty:
- `db` - obiekt bazy danych Chinook
- `llm` - model językowy Cohere (command-r-plus)

Następnie:
```python
context = toolkit.get_context()
tools = toolkit.get_tools()
```
Te dwie linie pobierają z zestawu narzędzi dwa krytyczne elementy:
- `context` - informacje o strukturze bazy danych (metadane), które będą dostarczone modelowi językowemu jako kontekst
- `tools` - konkretne narzędzia (funkcje), które model może wykorzystać do pracy z bazą danych

Ostatnia część kodu:
```python
print('**List of pre-defined Langchain Tools**')
print([tool.name for tool in tools])
print('')
print('**Context to pass to LLM on tables**')
print(context)
```
Służy do wyświetlenia tych informacji, co jest niezwykle pomocne w zrozumieniu, jak działa cały system.

Po uruchomieniu tego kodu zobaczymy:

1. **Listę predefiniowanych narzędzi LangChain** - prawdopodobnie zawierającą elementy takie jak:
   - `sql_db_query` - narzędzie do wykonywania zapytań SQL
   - `sql_db_schema` - narzędzie do przeglądania schematu bazy danych
   - `sql_db_list_tables` - narzędzie do listowania dostępnych tabel
   - `sql_db_describe_table` - narzędzie do szczegółowego opisu struktury konkretnej tabeli

2. **Kontekst przekazywany do modelu językowego** - będzie to obszerny tekst zawierający informacje o wszystkich tabelach, kolumnach, typach danych, kluczach obcych i innych relacjach w bazie danych Chinook. Kontekst ten jest niezwykle ważny, ponieważ zapewnia modelowi językowemu "mapy terenu" bazy danych - bez tych informacji model nie wiedziałby, jak formułować zapytania ani jakie dane są dostępne.

Ta część kodu pokazuje, jak w praktyce działa przygotowanie kontekstu dla modelu językowego w architekturze RAG (Retrieval-Augmented Generation). Zamiast polegać wyłącznie na wiedzy ogólnej modelu, system dostarcza mu specyficzne, kontekstowe informacje o bazie danych, na której ma operować. Jest to kluczowy element w budowaniu systemów AI zdolnych do pracy z własnymi, prywatnymi danymi organizacji.

Kod ten pełni również ważną funkcję edukacyjną i diagnostyczną - pozwala zobaczyć "pod maską" systemu i lepiej zrozumieć, jakie informacje są dostępne dla modelu językowego podczas generowania zapytań SQL.

In [ ]:
# define the prompt template
prompt = ChatPromptTemplate.from_template("{input}")

# instantiate the ReAct agent
agent = create_cohere_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt,
)
agent_executor = AgentExecutor(
    agent=agent, tools=tools, verbose=True, return_intermediate_steps=True
)

In [ ]:
agent_executor.invoke(
    {
        "input": "what tables are available?",
    }
)



> Entering new AgentExecutor chain...

I will use the tool to list the tables in the database.
{'tool_name': 'sql_db_list_tables', 'parameters': {'tool_input': ''}}
Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, TrackRelevant Documents: 0
Cited Documents: 0
Answer: The following tables are available: Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track.
Grounded answer: The following tables are available: <co: 0>Album</co: 0>, <co: 0>Artist</co: 0>, <co: 0>Customer</co: 0>, <co: 0>Employee</co: 0>, <co: 0>Genre</co: 0>, <co: 0>Invoice</co: 0>, <co: 0>InvoiceLine</co: 0>, <co: 0>MediaType</co: 0>, <co: 0>Playlist</co: 0>, <co: 0>PlaylistTrack</co: 0>, <co: 0>Track</co: 0>.

> Finished chain.


{'input': 'what tables are available?',
 'output': 'The following tables are available: Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track.',
 'citations': [CohereCitation(start=36, end=41, text='Album', documents=[{'output': 'Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track', 'id': 'doc_0'}], document_ids={'doc_0'}),
  CohereCitation(start=43, end=49, text='Artist', documents=[{'output': 'Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track', 'id': 'doc_0'}], document_ids={'doc_0'}),
  CohereCitation(start=51, end=59, text='Customer', documents=[{'output': 'Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track', 'id': 'doc_0'}], document_ids={'doc_0'}),
  CohereCitation(start=61, end=69, text='Employee', documents=[{'output': 'Album, Artist, Customer, Employee, Genre, Inv

In [ ]:
agent_executor.invoke(
    {
        "input": "show the first row of the Playlist and Genre tables?",
    }
)




> Entering new AgentExecutor chain...

I will use the sql_db_query tool to find the first row of the Playlist and Genre tables.
{'tool_name': 'sql_db_query', 'parameters': {'query': 'SELECT * FROM Playlist LIMIT 1;\nSELECT * FROM Genre LIMIT 1;'}}
Error: (sqlite3.ProgrammingError) You can only execute one statement at a time.
[SQL: SELECT * FROM Playlist LIMIT 1;
SELECT * FROM Genre LIMIT 1;]
(Background on this error at: https://sqlalche.me/e/20/f405)
I received an error message. I will now split the query into two separate queries.
{'tool_name': 'sql_db_query', 'parameters': {'query': 'SELECT * FROM Playlist LIMIT 1;'}}
[(1, 'Music')]
{'tool_name': 'sql_db_query', 'parameters': {'query': 'SELECT * FROM Genre LIMIT 1;'}}
[(1, 'Rock')]Relevant Documents: 2
Cited Documents: 2
Answer: The first row of the Playlist table is:
| ID | Name |
|---|---|
| 1 | Music |

The first row of the Genre table is:
| ID | Name |
|---|---|
| 1 | Rock |
Grounded answer: The first row of the Playlist tabl

{'input': 'show the first row of the Playlist and Genre tables?',
 'output': 'The first row of the Playlist table is:\n| ID | Name |\n|---|---|\n| 1 | Music |\n\nThe first row of the Genre table is:\n| ID | Name |\n|---|---|\n| 1 | Rock |',
 'citations': [CohereCitation(start=66, end=67, text='1', documents=[{'output': "[(1, 'Rock')]", 'id': 'doc_2'}], document_ids={'doc_2'}),
  CohereCitation(start=70, end=75, text='Music', documents=[{'output': "[(1, 'Rock')]", 'id': 'doc_2'}], document_ids={'doc_2'}),
  CohereCitation(start=142, end=143, text='1', documents=[{'output': "[(1, 'Rock')]", 'id': 'doc_2'}], document_ids={'doc_2'}),
  CohereCitation(start=146, end=150, text='Rock', documents=[{'output': "[(1, 'Rock')]", 'id': 'doc_2'}], document_ids={'doc_2'})],
 'intermediate_steps': [(AgentActionMessageLog(tool='sql_db_query', tool_input={'query': 'SELECT * FROM Playlist LIMIT 1;\nSELECT * FROM Genre LIMIT 1;'}, log="\nI will use the sql_db_query tool to find the first row of the Playli

In [ ]:
agent_executor.invoke(
    {
        "input": "which countries have the most invoices?",
    }
)



> Entering new AgentExecutor chain...

I will run a SQL query to find the countries with the most invoices.
{'tool_name': 'sql_db_query', 'parameters': {'query': 'SELECT country, COUNT(*) AS invoice_count FROM invoices GROUP BY country ORDER BY invoice_count DESC'}}
Error: (sqlite3.OperationalError) no such table: invoices
[SQL: SELECT country, COUNT(*) AS invoice_count FROM invoices GROUP BY country ORDER BY invoice_count DESC]
(Background on this error at: https://sqlalche.me/e/20/e3q8)
I received an error message. I will use the sql_db_list_tables tool to find the correct table name.
{'tool_name': 'sql_db_list_tables', 'parameters': {'tool_input': ''}}
Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track
The correct table name is 'Invoice'. I will now run the query again.
{'tool_name': 'sql_db_query', 'parameters': {'query': 'SELECT country, COUNT(*) AS invoice_count FROM Invoice GROUP BY country ORDER BY invoice_count DESC'}}
E

{'input': 'which countries have the most invoices?',
 'output': 'The countries with the most invoices are the USA (91), Canada (56), and France (35).',
 'citations': [CohereCitation(start=45, end=52, text='USA (91', documents=[{'output': "[('USA', 91), ('Canada', 56), ('France', 35), ('Brazil', 35), ('Germany', 28), ('United Kingdom', 21), ('Portugal', 14), ('Czech Republic', 14), ('India', 13), ('Sweden', 7), ('Spain', 7), ('Poland', 7), ('Norway', 7), ('Netherlands', 7), ('Italy', 7), ('Ireland', 7), ('Hungary', 7), ('Finland', 7), ('Denmark', 7), ('Chile', 7), ('Belgium', 7), ('Austria', 7), ('Australia', 7), ('Argentina', 7)]", 'id': 'doc_4'}], document_ids={'doc_4'}),
  CohereCitation(start=55, end=65, text='Canada (56', documents=[{'output': "[('USA', 91), ('Canada', 56), ('France', 35), ('Brazil', 35), ('Germany', 28), ('United Kingdom', 21), ('Portugal', 14), ('Czech Republic', 14), ('India', 13), ('Sweden', 7), ('Spain', 7), ('Poland', 7), ('Norway', 7), ('Netherlands', 7), ('

In [ ]:
output = agent_executor.invoke(
    {
        "input": "who is the best customer? The customer who has spent the most money is the best.",
    }
)



> Entering new AgentExecutor chain...

I will run a SQL query to find the customer who has spent the most money.
{'tool_name': 'sql_db_query', 'parameters': {'query': 'SELECT customer_name, SUM(order_total) AS total_spent FROM orders GROUP BY customer_name ORDER BY total_spent DESC LIMIT 1;'}}
Error: (sqlite3.OperationalError) no such table: orders
[SQL: SELECT customer_name, SUM(order_total) AS total_spent FROM orders GROUP BY customer_name ORDER BY total_spent DESC LIMIT 1;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)
I received an error message saying that there is no table called 'orders'. I will now use the sql_db_list_tables tool to find out what tables are available.
{'tool_name': 'sql_db_list_tables', 'parameters': {'tool_input': ''}}
Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track
I found that there is a table called 'Customer'. I will now use the sql_db_schema tool to find out what columns are in thi

In [ ]:
print(output["output"])

The best customer is Helena Holý, who has spent a total of 49.62.


# Reakcja z kontekstem

In [39]:
toolkit = SQLDatabaseToolkit(db=db, llm=llm)
context = toolkit.get_context()

Ten fragment kodu zajmuje się przygotowaniem kontekstu o strukturze bazy danych dla modelu językowego. To kluczowy element, który pozwala modelowi AI "widzieć" i rozumieć bazę danych, z którą będzie pracował.

Przyjrzyjmy się dokładniej, co dzieje się w tych dwóch liniach:

```python
toolkit = SQLDatabaseToolkit(db=db, llm=llm)
context = toolkit.get_context()
```

## Tworzenie zestawu narzędzi (toolkit)

W pierwszej linii tworzymy `SQLDatabaseToolkit` – specjalistyczny zestaw narzędzi, który działa jak pomost między modelem językowym a bazą danych. Jest to jak danie modelowi AI kompletnego zestawu narzędzi do pracy z konkretną bazą danych.

Przekazujemy do niego dwa kluczowe składniki:
- `db=db`: obiekt bazy danych Chinook, który zostały wcześniej utworzony. Ten obiekt zawiera wszystkie informacje o połączeniu z bazą.
- `llm=llm`: model językowy Cohere, który będzie "mózgiem" operacji, interpretującym pytania i formułującym odpowiedzi.

## Pobieranie kontekstu bazodanowego

W drugiej linii wywołujemy metodę `get_context()`, która wykonuje fascynujące zadanie za kulisami – generuje szczegółowy opis struktury bazy danych w formie tekstu. To jak stworzenie mapy bazy danych dla modelu językowego.

Metoda ta:
1. Analizuje bazę danych Chinook
2. Identyfikuje wszystkie tabele
3. Dla każdej tabeli spisuje wszystkie kolumny
4. Dla każdej kolumny notuje jej typ danych
5. Identyfikuje klucze główne i obce
6. Wykrywa relacje między tabelami
7. Organizuje wszystkie te informacje w ustrukturyzowany tekst

## Dlaczego to jest ważne

Wyobraź sobie, że prosisz kogoś o znalezienie informacji w bibliotece, której nigdy nie widział. Bez planu pięter, systemu katalogowania czy zrozumienia układu regałów, ta osoba byłaby całkowicie zagubiona. Dokładnie tak samo model językowy potrzebuje "mapy biblioteki" (bazy danych).

Kontekst wygenerowany przez `get_context()` daje modelowi AI precyzyjną wiedzę o:
- Jakie tabele są dostępne (np. Customers, Invoices, Tracks)
- Jakie informacje zawiera każda tabela (np. tabela Customers może zawierać FirstName, LastName, Email)
- Jak te tabele są ze sobą powiązane (np. Invoice ma CustomerID, który odnosi się do tabeli Customers)

Dzięki temu, gdy zapytamy "kto jest najlepszym klientem?", model potrafi:
1. Zidentyfikować, że informacje o klientach znajdują się w tabeli Customers
2. Zauważyć, że wydatki klientów są w tabeli Invoices
3. Zrozumieć, że tabele te są połączone przez pole CustomerID
4. Sformułować odpowiednie zapytanie SQL łączące te tabele

Zestaw narzędzi i pobrany kontekst razem stanowią fundament, na którym będzie operował agent AI, umożliwiając mu wykonywanie złożonych operacji na bazie danych w odpowiedzi na zapytania w języku naturalnym.

In [ ]:
print("Table Names")
print(context["table_names"])
print("Table Schemas")
print(context["table_info"])

Table Names
Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track
Table Schemas

CREATE TABLE "Album" (
	"AlbumId" INTEGER NOT NULL, 
	"Title" NVARCHAR(160) NOT NULL, 
	"ArtistId" INTEGER NOT NULL, 
	PRIMARY KEY ("AlbumId"), 
	FOREIGN KEY("ArtistId") REFERENCES "Artist" ("ArtistId")
)

/*
3 rows from Album table:
AlbumId	Title	ArtistId
1	For Those About To Rock We Salute You	1
2	Balls to the Wall	2
3	Restless and Wild	2
*/


CREATE TABLE "Artist" (
	"ArtistId" INTEGER NOT NULL, 
	"Name" NVARCHAR(120), 
	PRIMARY KEY ("ArtistId")
)

/*
3 rows from Artist table:
ArtistId	Name
1	AC/DC
2	Accept
3	Aerosmith
*/


CREATE TABLE "Customer" (
	"CustomerId" INTEGER NOT NULL, 
	"FirstName" NVARCHAR(40) NOT NULL, 
	"LastName" NVARCHAR(20) NOT NULL, 
	"Company" NVARCHAR(80), 
	"Address" NVARCHAR(70), 
	"City" NVARCHAR(40), 
	"State" NVARCHAR(40), 
	"Country" NVARCHAR(40), 
	"PostalCode" NVARCHAR(10), 
	"Phone" NVARCHAR(24), 
	"Fax" NVARCHAR(24), 
	"

In [ ]:
preamble = """## Task And Context
You use your advanced complex reasoning capabilities to help people by answering their questions and other requests interactively. You will be asked a very wide array of requests on all kinds of topics. You will be equipped with a wide range of search engines or similar tools to help you, which you use to research your answer. You may need to use multiple tools in parallel or sequentially to complete your task. You should focus on serving the user's needs as best you can, which will be wide-ranging.

## Style Guide
Unless the user asks for a different style of answer, you should answer in full sentences, using proper grammar and spelling.

## Additional Information
You are an expert who answers the user's question by creating SQL queries and executing them.
You are equipped with a number of relevant SQL tools.

Here is information about the database:
{schema_info}
""".format(schema_info=context)

In [ ]:
output = agent_executor.invoke(
    {
        "input": "provide the name of the best customer? The customer who has spent the most money is the best.",
        "preamble": preamble,
    }
)



> Entering new AgentExecutor chain...

I will first find the total amount spent by each customer, then I will find the customer with the highest total amount spent.
{'tool_name': 'sql_db_query', 'parameters': {'query': 'SELECT CustomerId, SUM(Total) AS TotalSpent FROM Invoice GROUP BY CustomerId ORDER BY TotalSpent DESC'}}
[(6, 49.620000000000005), (26, 47.620000000000005), (57, 46.62), (45, 45.62), (46, 45.62), (28, 43.620000000000005), (24, 43.62), (37, 43.62), (7, 42.62), (25, 42.62), (44, 41.620000000000005), (5, 40.620000000000005), (43, 40.620000000000005), (48, 40.62), (17, 39.620000000000005), (34, 39.620000000000005), (1, 39.62), (3, 39.62), (4, 39.62), (20, 39.62), (22, 39.62), (42, 39.62), (15, 38.620000000000005), (19, 38.620000000000005), (39, 38.620000000000005), (40, 38.620000000000005), (51, 38.620000000000005), (58, 38.620000000000005), (2, 37.620000000000005), (9, 37.620000000000005), (10, 37.620000000000005), (11, 37.620000000000005), (13, 37.620000000000005), (14,

In [ ]:
print(output["output"])

The customer who has spent the most money is the one with ID 6, who has spent a total of 49.62.
